In [5]:
import pandas as pd
import numpy as np


In [4]:

# Load the real dataset directly from the data folder
print("Loading dataset...")
df = pd.read_csv('data/HI-Small_Trans.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])


Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'data/HI-Small_Trans.csv'

In [3]:

# 1. Cross-Border & Cross-Currency Flags
print("Engineering cross-border and currency flags...")
df['is_cross_border'] = (df['From Bank'] != df['To Bank']).astype(int)
df['is_cross_currency'] = (df['Receiving Currency'] != df['Payment Currency']).astype(int)


Engineering cross-border and currency flags...


NameError: name 'df' is not defined

In [ ]:

# 2. Account-level Aggregations
print("Calculating account-level aggregations...")
tx_counts = df.groupby('Account').size().reset_index(name='tx_count_out')
df = df.merge(tx_counts, on='Account', how='left')

unique_counterparties = df.groupby('Account')['Account.1'].nunique().reset_index(name='unique_counterparties_out')
df = df.merge(unique_counterparties, on='Account', how='left')


In [ ]:

# 3. Rolling Time Windows (Burstiness)
print("Computing rolling window burstiness...")
df = df.sort_values('Timestamp')
df_indexed = df.set_index('Timestamp')
rolling_counts = df_indexed.groupby('Account').rolling('24h')['Amount Paid'].count().reset_index(name='tx_24h_burst')
df = df.merge(rolling_counts, on=['Account', 'Timestamp'], how='left')


In [ ]:

# 4. Round-Number Flags
print("Flagging round amounts...")
df['is_round_amount'] = (df['Amount Paid'] % 1000 == 0).astype(int)


In [ ]:

# 5. In/Out Ratio
print("Calculating in/out ratios...")
account_totals = df.groupby('Account').agg(
    total_paid=('Amount Paid', 'sum'),
    total_received=('Amount Received', 'sum')
).reset_index()
account_totals['in_out_ratio'] = account_totals['total_received'] / (account_totals['total_paid'] + 1)
df = df.merge(account_totals[['Account', 'in_out_ratio']], on='Account', how='left')


In [ ]:

# 6. Fill missing values that might result from rolling windows
df.fillna(0, inplace=True)


In [ ]:

# 7. Export the Final Feature Table
print("Exporting feature table...")
df.to_parquet("data/features_stage2.parquet", index=False)
print("Feature Engineering Complete. Table saved to data/features_stage2.parquet ready for Person 3.")